# Synthetic predictive-maintenance walkthrough

All data and models here are independent fictional examples. Metrics describe an invented simulation and do not establish field accuracy. See the model card and README.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd()/'maintenance_app').exists() else Path.cwd().parent
APP = ROOT/'maintenance_app'
sys.path.insert(0, str(APP/'data/model'))
sys.path.insert(0, str(APP))
import maintenance_ml as ml
import forecasting as fc
import pandas as pd


## Inspect the generated data

The source generator and fixed seed are public. Equipment and station identifiers are fictional.


In [ ]:
data = APP/'data/demo'
assets = pd.read_csv(data/'assets.csv')
history = pd.read_csv(data/'observations.csv')
history[ml.TIME] = pd.to_datetime(history[ml.TIME])
assets.head()


In [ ]:
pd.read_csv(data/'events.csv').groupby('Maintenance Type').size()


## Replay a dated forecast

Risk applies after the displayed observation date. It does not invent a new measurement or an exact failure date.


In [ ]:
engine = fc.ForecastEngine(APP/'data/forecast_models')
frame, _ = fc.build_features(history, assets, engine.events)
current = frame[frame[ml.ID].eq('DEMO-A-001')].tail(1)
result = engine.predict_frame(current, policy='catch_more', explain_horizon=7)[0]
print('Observation:', current.iloc[0][ml.TIME])
pd.DataFrame(result['risks'])[['days','probability','threshold','inspect','horizon_ends']]


In [ ]:
pd.DataFrame(result['drivers'])


## Review held-out synthetic results

Parameters are fixed. Calibration and threshold selection use separate earlier periods. Test windows overlap and can describe the same failure. Approved schedules are independent of these scores.


In [ ]:
report = json.loads((APP/'data/forecast_models/evaluation.json').read_text())
pd.DataFrame(report['horizons'])[['horizon_days','n','failures','recall','precision','alert_fraction','roc_auc']]
